# Célula 1: Configuração e Importações

In [1]:
import os, random, time
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import pywt
from google.colab import drive
from xgboost import XGBClassifier
from sklearn.svm import OneClassSVM
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# Configuração de Reproduzibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Montar Google Drive
drive.mount('/content/drive')

print(f"Ambiente configurado. GPU: {torch.cuda.is_available()}")

Mounted at /content/drive
Ambiente configurado. GPU: True


# Célula 2: Leitura de Áudios (Google Drive)

Ajustar o BASE_PATH para o local onde você salvou suas pastas /normal e /anomaly.

In [3]:
def load_drive_dataset(base_path, sr=16000, duration=16):
    data, labels = [], []
    class_map = {'normal': 0, 'anomaly': 1}

    for label_name, label_val in class_map.items():
        folder_path = os.path.join(base_path, label_name)
        if not os.path.exists(folder_path):
            print(f"Aviso: Pasta {label_name} não encontrada em {folder_path}")
            continue

        files = [f for f in os.listdir(folder_path) if f.endswith(('.wav', '.mp3'))]
        print(f"Carregando {len(files)} arquivos de: {label_name}")

        for file in files:
            file_path = os.path.join(folder_path, file)
            audio, _ = librosa.load(file_path, sr=sr, duration=duration)
            if len(audio) < sr * duration:
                audio = np.pad(audio, (0, int(sr * duration) - len(audio)))
            data.append(audio)
            labels.append(label_val)

    return np.array(data), np.array(labels)

# AJUSTE ESTE CAMINHO:
BASE_PATH = '/content/drive/MyDrive/PROJETO APLICADO - Acoustic Anomaly Detection (AAD)/Arquivos de audio'
X_raw, y_raw = load_drive_dataset(BASE_PATH)

Carregando 1 arquivos de: normal
Carregando 1 arquivos de: anomaly


Célula 3: Janelamento (Sliding Window)

In [4]:
def apply_sliding_window(X, y, window_size=2, overlap=0.5, sr=16000):
    step = int(window_size * sr * (1 - overlap))
    w_len = int(window_size * sr)
    all_segments, all_labels = [], []

    for i in range(len(X)):
        audio = X[i]
        label = y[i]
        for start in range(0, len(audio) - w_len + 1, step):
            all_segments.append(audio[start : start + w_len])
            all_labels.append(label)

    return np.array(all_segments), np.array(all_labels)

X_win, y_win = apply_sliding_window(X_raw, y_raw)
print(f"Janelas totais: {len(X_win)}")

Janelas totais: 30


# Célula 4: Extração de Features (Multi-Model)

Aqui preparamos os dados para os dois mundos: Deep Learning (2D) e Estatístico (1D).

In [5]:
def extract_features(X_segments, sr=16000):
    feats_mel = [] # Para CNN
    feats_stat = [] # Para XGBoost e OCSVM

    for audio in X_segments:
        # Mel Spectrogram (64x63 pixels aprox)
        mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=64)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        feats_mel.append(mel_db[np.newaxis, ...])

        # MFCC + Estatísticas (Vetor de características)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)
        feats_stat.append(np.concatenate([mfcc_mean, mfcc_std]))

    return np.array(feats_mel), np.array(feats_stat)

X_mel, X_stat = extract_features(X_win)

Célula 5: Definição dos Modelos

In [6]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.fc = nn.Sequential(nn.Linear(32 * 16 * 15, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x): return self.fc(self.conv(x))

# Instanciação
cnn_model = TinyCNN().to(DEVICE)
xgb_model = XGBClassifier(scale_pos_weight=(len(y_win)-sum(y_win))/sum(y_win)) # Ajuste de desbalanceamento
ocsvm_model = OneClassSVM(nu=0.1, kernel="rbf", gamma='auto')

# Célula 6: Treinamento e Split

In [7]:
# Split mantendo a proporção para todos
idx = np.arange(len(y_win))
train_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=y_win, random_state=SEED)

# Treino XGBoost e OCSVM (usam apenas dados normais para o OCSVM se quiser ser purista)
xgb_model.fit(X_stat[train_idx], y_win[train_idx])
ocsvm_model.fit(X_stat[train_idx][y_win[train_idx] == 0]) # Treina apenas no "Normal"

# Treino CNN
train_ds = DataLoader(TensorDataset(torch.tensor(X_mel[train_idx]).float(), torch.tensor(y_win[train_idx])), batch_size=32, shuffle=True)
opt = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
crit = nn.CrossEntropyLoss()

cnn_model.train()
for epoch in range(15):
    for xb, yb in train_ds:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); crit(cnn_model(xb), yb).backward(); opt.step()

# Célula 7: Benchmark Final e Explicação das Métricas

Aqui consolidamos tudo para a decisão técnica.

In [8]:
def run_benchmark():
    # 1. CNN
    cnn_model.eval()
    with torch.no_grad():
        out = cnn_model(torch.tensor(X_mel[test_idx]).to(DEVICE))
        prob_cnn = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        pred_cnn = np.argmax(out.cpu().numpy(), axis=1)

    # 2. XGBoost
    prob_xgb = xgb_model.predict_proba(X_stat[test_idx])[:, 1]
    pred_xgb = xgb_model.predict(X_stat[test_idx])

    # 3. OCSVM
    score_oc = -ocsvm_model.decision_function(X_stat[test_idx])
    pred_oc = np.where(ocsvm_model.predict(X_stat[test_idx]) == -1, 1, 0)

    # Consolidação
    data = []
    for name, pred, prob in [("CNN", pred_cnn, prob_cnn), ("XGBoost", pred_xgb, prob_xgb), ("OC-SVM", pred_oc, score_oc)]:
        data.append({
            "Modelo": name,
            "Accuracy": accuracy_score(y_win[test_idx], pred),
            "Precision (Evita Falso Alarme)": precision_score(y_win[test_idx], pred),
            "Recall (Não perde falha)": recall_score(y_win[test_idx], pred),
            "F1-Score (Equilíbrio)": f1_score(y_win[test_idx], pred),
            "pAUC (FPR < 0.1)": roc_auc_score(y_win[test_idx], prob, max_fpr=0.1)
        })
    return pd.DataFrame(data)

display(run_benchmark())

,Modelo,Accuracy,Precision (Evita Falso Alarme),Recall (Não perde falha),F1-Score (Equilíbrio),pAUC (FPR < 0.1)
0,CNN,1.0,1.0,1.0,1.000000,1.0
1,XGBoost,1.0,1.0,1.0,1.000000,1.0
2,OC-SVM,0.5,0.5,1.0,0.666667,1.0
